# 🚀 SkyOBS Studio — 100% Free 24/7 Cloud Live Streamer
### Stream 4K/1080p Videos to YouTube Live from any Phone or Browser with ZERO Laptop Dependency

**How to use:**
1. Click the **Play button ▶** on the code cell below.
2. Wait ~20 seconds — a green button and public HTTPS link (`https://xxxx.trycloudflare.com`) will appear.
3. Open the link on your iPhone or Android phone.
4. Upload your video directly from your phone camera roll / storage, paste your YouTube Stream Key, pick 4K or 1080p, and tap **START STREAMING**!
5. **You can turn off your laptop, close the lid, or disconnect — the cloud server runs independently!**

In [ ]:
# @title ⚡ 1-CLICK LAUNCH: START 100% FREE CLOUD SERVER (NO LAPTOP NEEDED!)
import os, sys, subprocess, time, re, io, base64, zipfile, requests

print('[1/5] Extracting SkyOBS Studio application files...')
os.makedirs('/content/skyobs', exist_ok=True)
BUNDLE_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as zf:
    zf.extractall('/content/skyobs')

print('[2/5] Installing FFmpeg and cloud Python dependencies...')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'curl'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'pydantic', 'python-multipart', 'requests'], check=True)

print('[3/5] Downloading Cloudflare high-speed tunnel...')
subprocess.run(['curl', '-L', '-o', '/usr/local/bin/cloudflared', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

print('[4/5] Starting SkyOBS Engine on Cloud port 7860...')
env = os.environ.copy()
env['PORT'] = '7860'
env['HOST'] = '0.0.0.0'
server_proc = subprocess.Popen([sys.executable, 'app.py', '--server'], cwd='/content/skyobs', env=env)

# Wait and verify backend is 100% healthy before exposing
is_ready = False
for attempt in range(25):
    time.sleep(1)
    try:
        r = requests.get('http://127.0.0.1:7860/api/status', timeout=1)
        if r.status_code == 200:
            is_ready = True
            break
    except Exception:
        pass

if not is_ready:
    print('[!] Error: Engine startup timed out. Check logs.')
    sys.exit(1)

print('[5/5] Engine is HEALTHY! Generating secure Public Cloud Link...')
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tun.stderr.readline()
    if not line:
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print('\n' + '='*70)
    print('  🎉 100% FREE CLOUD SERVER IS LIVE AND ACCESSIBLE WORLDWIDE!')
    print(f'  👉 PUBLIC LINK: {public_url}')
    print('='*70)
    print('\n>> Open this link on your iPhone, Android, or share with anyone!')
    print('>> You can CLOSE and TURN OFF your laptop now — Google Cloud handles the stream!')
    try:
        from IPython.display import HTML, display
        display(HTML(f'''
        <div style="background:#161920; border:2px solid #00e676; border-radius:12px; padding:20px; margin:15px 0; font-family:sans-serif;">
            <h2 style="color:#00e676; margin:0 0 10px 0;">🚀 SkyOBS Studio Cloud is Online!</h2>
            <p style="color:#e0e0e0; font-size:16px; margin:0 0 15px 0;">Tap the button below to open SkyOBS on your phone or browser:</p>
            <a href="{public_url}" target="_blank" style="background:#00e676; color:#000; font-weight:bold; font-size:18px; padding:12px 24px; text-decoration:none; border-radius:8px; display:inline-block;">👉 OPEN SKYOBS STUDIO ON PHONE</a>
            <p style="color:#888; font-size:13px; margin:12px 0 0 0;">✨ Works 24/7 on Google Cloud even when your laptop is turned off.</p>
        </div>
        '''))
    except Exception:
        pass
    tun.wait()
else:
    print('[!] Failed to acquire public URL. Check internet connection.')
